<div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 30px; border-radius: 10px; color: white;'>
    <h1 style='margin: 0; font-size: 36px;'>🔍 Explainability & Ensembling</h1>
    <h3 style='margin: 10px 0 0 0; font-weight: 300;'>Making Models Transparent and Powerful</h3>
    <p style='margin: 15px 0 0 0; opacity: 0.9;'>Vishlesan i-Hub IIT Patna × Masai School — AIM Program</p>
    <div style='margin-top: 20px; display: flex; gap: 20px; font-size: 14px;'>
        <span>⏱️ Duration: 110 minutes</span>
        <span>📊 Difficulty: Intermediate</span>
        <span>🎯 Hands-on: 4 modules</span>
    </div>
</div>

## 🎯 Learning Objectives

| Objective | Description |
|-----------|-------------|
| **1. SHAP Fundamentals** | Understand SHAP values using game theory intuition and Shapley value concepts |
| **2. Global Explanations** | Interpret model behavior using SHAP summary plots and mean absolute SHAP bar charts |
| **3. Local Explanations** | Analyze individual predictions using SHAP waterfall plots for instance-level insights |
| **4. Simple Ensembling** | Implement model averaging and weighted blending for improved predictions |
| **5. Stacking Concept** | Understand meta-model approaches and why stacking outperforms simple averaging |

<div style='background: #fff3cd; padding: 15px; border-left: 5px solid #ffc107; border-radius: 5px; margin: 20px 0;'>
    <h4 style='margin: 0 0 10px 0; color: #856404;'>📋 Business Context</h4>
    <p style='margin: 0; color: #856404;'>
        <strong>Scenario:</strong> You're a data scientist at CreditFlow, a fintech company. Your XGBoost model achieves 94% accuracy in predicting loan defaults, but:
        <br><br>
        <strong>Problem 1:</strong> Regulators require explanations for loan rejections (EU GDPR Article 22)<br>
        <strong>Problem 2:</strong> Business wants to maximize accuracy while maintaining trust<br>
        <strong>Problem 3:</strong> Some high-value customers are incorrectly rejected and you need to know why
        <br><br>
        <strong>Your mission:</strong> Make your model interpretable using SHAP and improve accuracy through ensembling.
    </p>
</div>

## 📚 Session Roadmap

<div style='background: #e3f2fd; padding: 15px; border-radius: 5px; margin: 20px 0;'>
    <table style='width: 100%; border-collapse: collapse;'>
        <tr style='background: #1976d2; color: white;'>
            <th style='padding: 10px; text-align: left;'>Module</th>
            <th style='padding: 10px; text-align: left;'>Topic</th>
            <th style='padding: 10px; text-align: center;'>Duration</th>
        </tr>
        <tr style='background: white;'>
            <td style='padding: 10px;'>Module 1</td>
            <td style='padding: 10px;'><strong>Foundation:</strong> Setup & Base Models</td>
            <td style='padding: 10px; text-align: center;'>25 min</td>
        </tr>
        <tr style='background: #f5f5f5;'>
            <td style='padding: 10px;'>Module 2</td>
            <td style='padding: 10px;'><strong>SHAP Interpretability:</strong> Global & Local Explanations</td>
            <td style='padding: 10px; text-align: center;'>35 min</td>
        </tr>
        <tr style='background: white;'>
            <td style='padding: 10px;'>Module 3</td>
            <td style='padding: 10px;'><strong>Model Ensembling:</strong> Averaging, Weighting, Stacking</td>
            <td style='padding: 10px; text-align: center;'>40 min</td>
        </tr>
        <tr style='background: #f5f5f5;'>
            <td style='padding: 10px;'>Module 4</td>
            <td style='padding: 10px;'><strong>Conclusion:</strong> Results & Best Practices</td>
            <td style='padding: 10px; text-align: center;'>10 min</td>
        </tr>
    </table>
</div>

<div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 10px; color: white; margin: 30px 0;'>
    <h2 style='margin: 0; font-size: 28px;'>MODULE 1: Foundation - Setup & Base Models</h2>
    <p style='margin: 10px 0 0 0; opacity: 0.9;'>Building three diverse models for loan default prediction</p>
    <p style='margin: 5px 0 0 0; font-size: 14px;'>⏱️ Duration: 25 minutes</p>
</div>

### 1.1 Import Libraries & Setup

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Models
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# SHAP for interpretability
import shap

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")
print(f"   SHAP version: {shap.__version__}")

### 1.2 Load and Prepare Credit Risk Dataset

In [ ]:
# Generate synthetic credit risk dataset
np.random.seed(42)
n_samples = 2000

# Generate features
credit_score = np.random.normal(700, 80, n_samples).clip(300, 850)
annual_income = np.random.exponential(50000, n_samples).clip(20000, 200000)
debt_to_income = np.random.beta(2, 5, n_samples) * 100
employment_length = np.random.poisson(5, n_samples).clip(0, 30)
age = np.random.normal(40, 12, n_samples).clip(22, 70)
num_credit_lines = np.random.poisson(3, n_samples).clip(0, 15)

# Create target (loan default: 0=no default, 1=default)
# Higher credit score, income → lower default probability
# Higher debt-to-income → higher default probability
default_prob = (
    -0.003 * credit_score +
    -0.000008 * annual_income +
    0.02 * debt_to_income +
    -0.01 * employment_length +
    -0.005 * age +
    0.02 * num_credit_lines +
    3.0
)
default_prob = 1 / (1 + np.exp(-default_prob))
loan_default = (np.random.random(n_samples) < default_prob).astype(int)

# Create DataFrame
df = pd.DataFrame({
    'credit_score': credit_score,
    'annual_income': annual_income,
    'debt_to_income_ratio': debt_to_income,
    'employment_length_years': employment_length,
    'age': age,
    'num_credit_lines': num_credit_lines,
    'loan_default': loan_default
})

print("="*60)
print("📊 CREDIT RISK DATASET")
print("="*60)
print(f"Total samples: {len(df):,}")
print(f"Default rate: {df['loan_default'].mean():.1%}")
print("\nFeature summary:")
print(df.describe().round(2))
print("\nClass distribution:")
print(df['loan_default'].value_counts())

### 1.3 Train-Test Split

In [ ]:
# Separate features and target
X = df.drop('loan_default', axis=1)
y = df['loan_default']

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("="*60)
print("✂️ TRAIN-TEST SPLIT")
print("="*60)
print(f"Training set:   {len(X_train):,} samples ({len(X_train)/len(df):.0%})")
print(f"Test set:       {len(X_test):,} samples ({len(X_test)/len(df):.0%})")
print(f"\nFeatures: {list(X.columns)}")
print(f"\nTrain default rate: {y_train.mean():.1%}")
print(f"Test default rate:  {y_test.mean():.1%}")

### 1.4 Train Three Base Models

<div style='background: #e1f5fe; padding: 15px; border-left: 5px solid #0288d1; border-radius: 5px; margin: 15px 0;'>
    <h4 style='margin: 0 0 10px 0; color: #01579b;'>💡 Why Three Different Models?</h4>
    <p style='margin: 0; color: #01579b;'>
        <strong>Diversity is key to ensemble success!</strong><br><br>
        • <strong>XGBoost:</strong> Captures complex non-linear patterns, great with interactions<br>
        • <strong>Random Forest:</strong> Robust to outliers, handles different feature scales well<br>
        • <strong>Logistic Regression:</strong> Fast, interpretable, captures linear relationships<br><br>
        Each model makes <em>different errors</em> → Combined predictions are more robust!
    </p>
</div>

In [ ]:
# Model 1: XGBoost
print("Training XGBoost...")
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]
xgb_acc = accuracy_score(y_test, xgb_pred)
xgb_auc = roc_auc_score(y_test, xgb_prob)

# Model 2: Random Forest
print("Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]
rf_acc = accuracy_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

# Model 3: Logistic Regression
print("Training Logistic Regression...")
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
lr_prob = lr_model.predict_proba(X_test)[:, 1]
lr_acc = accuracy_score(y_test, lr_pred)
lr_auc = roc_auc_score(y_test, lr_prob)

# Create results DataFrame
results = pd.DataFrame({
    'Model': ['XGBoost', 'Random Forest', 'Logistic Regression'],
    'Accuracy': [xgb_acc, rf_acc, lr_acc],
    'AUC-ROC': [xgb_auc, rf_auc, lr_auc]
})

print("\n" + "="*60)
print("📈 BASE MODEL PERFORMANCE")
print("="*60)
print(results.to_string(index=False))
print(f"\nBest single model: {results.loc[results['Accuracy'].idxmax(), 'Model']}")
print(f"Accuracy: {results['Accuracy'].max():.4f}")

<div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 10px; color: white; margin: 30px 0;'>
    <h2 style='margin: 0; font-size: 28px;'>MODULE 2: SHAP Interpretability</h2>
    <p style='margin: 10px 0 0 0; opacity: 0.9;'>Understanding model decisions through game theory</p>
    <p style='margin: 5px 0 0 0; font-size: 14px;'>⏱️ Duration: 35 minutes</p>
</div>

### 2.1 SHAP: The Game Theory Intuition

<div style='background: #f3e5f5; padding: 15px; border-left: 5px solid #9c27b0; border-radius: 5px; margin: 15px 0;'>
    <h4 style='margin: 0 0 10px 0; color: #4a148c;'>🎮 Game Theory Analogy</h4>
    <p style='margin: 0; color: #4a148c;'>
        <strong>Soccer Team Example:</strong><br><br>
        Team wins 3-0 with all 11 players. How much credit does each player deserve?<br><br>
        <strong>SHAP approach:</strong><br>
        • Play game with striker removed → Only 1 goal scored<br>
        • Striker's contribution: 3 - 1 = <strong>2 goals</strong><br>
        • Repeat for all players across all possible team combinations<br>
        • Average contribution = Shapley value (provably fair!)<br><br>
        <strong>ML translation:</strong><br>
        Players → Features | Goals → Prediction value | Team → Model
    </p>
</div>

### 2.2 Calculate SHAP Values for XGBoost

In [ ]:
# Create SHAP explainer for tree-based model
print("Creating SHAP explainer for XGBoost...")
explainer = shap.TreeExplainer(xgb_model)

# Calculate SHAP values for test set
print("Computing SHAP values (this may take a moment)...")
shap_values = explainer.shap_values(X_test)

print("\n" + "="*60)
print("✅ SHAP VALUES COMPUTED")
print("="*60)
print(f"Shape of SHAP values: {shap_values.shape}")
print(f"  - Rows: {shap_values.shape[0]} test samples")
print(f"  - Columns: {shap_values.shape[1]} features")
print(f"\nBase value (average prediction): {explainer.expected_value:.4f}")
print(f"\nSHAP values represent: How much each feature pushes prediction")
print(f"  away from the base value for each sample")

### 2.3 Global Explanation: SHAP Summary Plot

<div style='background: #e8f5e9; padding: 15px; border-left: 5px solid #4caf50; border-radius: 5px; margin: 15px 0;'>
    <h4 style='margin: 0 0 10px 0; color: #1b5e20;'>📊 How to Read SHAP Summary Plot</h4>
    <p style='margin: 0; color: #1b5e20;'>
        <strong>X-axis:</strong> SHAP value (impact on prediction)<br>
        <strong>Y-axis:</strong> Features (sorted by importance)<br>
        <strong>Color:</strong> Feature value (red = high, blue = low)<br><br>
        <strong>Example interpretation:</strong><br>
        • Red dots on right → High feature value increases prediction<br>
        • Blue dots on left → Low feature value decreases prediction<br>
        • Wide spread → Feature impact varies greatly across samples
    </p>
</div>

In [ ]:
# Create SHAP summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, plot_type="dot", show=False)
plt.title("SHAP Summary Plot - Feature Impact Distribution", fontsize=14, pad=15)
plt.tight_layout()
plt.show()

print("="*60)
print("🔍 KEY INSIGHTS FROM SUMMARY PLOT")
print("="*60)
print("1. Most important features are at the top")
print("2. Color shows feature value: RED (high) vs BLUE (low)")
print("3. Position shows SHAP value: RIGHT (increases default risk)")
print("                              LEFT (decreases default risk)")
print("\nExpected pattern:")
print("  • High debt_to_income (red) → Right (increases default)")
print("  • High credit_score (red) → Left (decreases default)")

### 2.4 Global Explanation: Mean Absolute SHAP Bar Chart

In [ ]:
# Calculate mean absolute SHAP values for each feature
mean_abs_shap = np.abs(shap_values).mean(axis=0)

# Create DataFrame for visualization
feature_importance = pd.DataFrame({
    'Feature': X_test.columns,
    'Mean |SHAP|': mean_abs_shap
}).sort_values('Mean |SHAP|', ascending=False)

# Plot
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Mean |SHAP|'], color='#667eea')
plt.xlabel('Mean |SHAP value|', fontsize=12)
plt.title('Feature Importance - Mean Absolute SHAP Values', fontsize=14, pad=15)
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("="*60)
print("📊 FEATURE IMPORTANCE RANKING")
print("="*60)
print(feature_importance.to_string(index=False))

# Calculate cumulative importance
feature_importance['Cumulative %'] = (
    feature_importance['Mean |SHAP|'].cumsum() / 
    feature_importance['Mean |SHAP|'].sum() * 100
)

print("\n" + "="*60)
print("🎯 CUMULATIVE FEATURE IMPORTANCE")
print("="*60)
for idx, row in feature_importance.iterrows():
    print(f"Top {feature_importance.index.get_loc(idx)+1}: {row['Cumulative %']:.1f}% of total impact")

### 2.5 Local Explanation: SHAP Waterfall Plot

<div style='background: #fff3e0; padding: 15px; border-left: 5px solid #ff9800; border-radius: 5px; margin: 15px 0;'>
    <h4 style='margin: 0 0 10px 0; color: #e65100;'>🔎 Local Explanation Purpose</h4>
    <p style='margin: 0; color: #e65100;'>
        <strong>Global:</strong> "What matters most across ALL predictions?"<br>
        <strong>Local:</strong> "Why THIS specific prediction?"<br><br>
        <strong>Use cases:</strong><br>
        • Explain loan rejection to specific customer<br>
        • Debug unexpected model behavior<br>
        • Counterfactual: "What would change the decision?"<br>
        • Regulatory compliance (GDPR right to explanation)
    </p>
</div>

In [ ]:
# Select an interesting sample (high-risk applicant)
sample_idx = np.argmax(xgb_prob)  # Highest predicted default probability

print("="*60)
print(f"📋 APPLICANT #{sample_idx} - DETAILED EXPLANATION")
print("="*60)
print("\nApplicant Profile:")
for feature, value in X_test.iloc[sample_idx].items():
    print(f"  • {feature:30s}: {value:8.2f}")

print(f"\nModel Prediction:")
print(f"  Default probability: {xgb_prob[sample_idx]:.1%}")
print(f"  Decision: {'REJECT (High Risk)' if xgb_pred[sample_idx]==1 else 'APPROVE (Low Risk)'}")
print(f"  Actual outcome: {'Defaulted' if y_test.iloc[sample_idx]==1 else 'Did not default'}")

In [ ]:
# Create waterfall plot for this sample
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[sample_idx],
        base_values=explainer.expected_value,
        data=X_test.iloc[sample_idx].values,
        feature_names=X_test.columns.tolist()
    ),
    show=False
)
plt.title(f"SHAP Waterfall - Applicant #{sample_idx}", fontsize=14, pad=15)
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("💡 HOW TO READ WATERFALL PLOT")
print("="*60)
print("• Start with base value (average model prediction)")
print("• Each feature pushes prediction UP (red) or DOWN (blue)")
print("• Arrow length = feature impact magnitude")
print("• Final value = actual prediction for this applicant")
print("\nCommunication to customer:")
print("  'Your application was rejected primarily due to:'")
print("  • [List top negative contributors from waterfall]")

### 💪 Exercise: Analyze Another Sample

**Task:** Find an applicant who was APPROVED but actually DEFAULTED (false negative)

In [ ]:
# Find false negatives (predicted 0, actual 1)
false_negatives = np.where((xgb_pred == 0) & (y_test == 1))[0]

if len(false_negatives) > 0:
    fn_idx = false_negatives[0]
    
    print("="*60)
    print(f"⚠️ FALSE NEGATIVE ANALYSIS - Applicant #{fn_idx}")
    print("="*60)
    print("Predicted: LOW RISK (approved) ✓")
    print("Actual: DEFAULTED ✗")
    print("\nApplicant Profile:")
    for feature, value in X_test.iloc[fn_idx].items():
        print(f"  • {feature:30s}: {value:8.2f}")
    
    # Create waterfall
    shap.plots.waterfall(
        shap.Explanation(
            values=shap_values[fn_idx],
            base_values=explainer.expected_value,
            data=X_test.iloc[fn_idx].values,
            feature_names=X_test.columns.tolist()
        ),
        show=False
    )
    plt.title(f"Why was this applicant approved? (False Negative)", fontsize=14, pad=15)
    plt.tight_layout()
    plt.show()
    
    print("\n🔍 Model debugging insight:")
    print("  Look for features that incorrectly pushed toward approval")
else:
    print("No false negatives found in test set!")

<div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 10px; color: white; margin: 30px 0;'>
    <h2 style='margin: 0; font-size: 28px;'>MODULE 3: Model Ensembling</h2>
    <p style='margin: 10px 0 0 0; opacity: 0.9;'>Combining models for superior predictions</p>
    <p style='margin: 5px 0 0 0; font-size: 14px;'>⏱️ Duration: 40 minutes</p>
</div>

### 3.1 Why Ensemble? Error Diversity

<div style='background: #e1f5fe; padding: 15px; border-left: 5px solid #0288d1; border-radius: 5px; margin: 15px 0;'>
    <h4 style='margin: 0 0 10px 0; color: #01579b;'>🎯 Ensemble Intuition</h4>
    <p style='margin: 0; color: #01579b;'>
        <strong>Single model:</strong> Makes specific errors based on its biases<br>
        <strong>Ensemble:</strong> Errors cancel out when models disagree!<br><br>
        <strong>Example:</strong><br>
        Sample #47: XGBoost says 0.8, RF says 0.3, LR says 0.6<br>
        Average: 0.57 (middle ground, more robust)<br><br>
        <strong>Key requirement:</strong> Models must be DIVERSE (make different errors)
    </p>
</div>

In [ ]:
# Analyze error correlation between models
errors_df = pd.DataFrame({
    'XGB_error': (xgb_pred != y_test).astype(int),
    'RF_error': (rf_pred != y_test).astype(int),
    'LR_error': (lr_pred != y_test).astype(int)
})

# Correlation of errors
error_corr = errors_df.corr()

print("="*60)
print("🔍 ERROR CORRELATION ANALYSIS")
print("="*60)
print("\nError correlation matrix:")
print(error_corr.round(3))
print("\n💡 Interpretation:")
print("  • Low correlation (<0.5): Models make DIFFERENT errors ✓")
print("  • High correlation (>0.8): Models make SAME errors ✗")
print("\nOur models:")
avg_corr = error_corr.values[np.triu_indices_from(error_corr.values, k=1)].mean()
print(f"  Average error correlation: {avg_corr:.3f}")
print(f"  Status: {'Good diversity!' if avg_corr < 0.6 else 'Models too similar'}")

# Count disagreements
all_agree = (errors_df.sum(axis=1) == 0) | (errors_df.sum(axis=1) == 3)
print(f"\n📊 Agreement statistics:")
print(f"  All models agree: {all_agree.sum()}/{len(y_test)} samples ({all_agree.mean():.1%})")
print(f"  At least one disagrees: {(~all_agree).sum()} samples ({(~all_agree).mean():.1%})")
print(f"\n  → Ensemble can potentially correct {(~all_agree).sum()} predictions!")

### 3.2 Method 1: Simple Averaging

In [ ]:
# Simple averaging of probabilities
avg_prob = (xgb_prob + rf_prob + lr_prob) / 3
avg_pred = (avg_prob >= 0.5).astype(int)

avg_acc = accuracy_score(y_test, avg_pred)
avg_auc = roc_auc_score(y_test, avg_prob)

print("="*60)
print("📊 SIMPLE AVERAGING RESULTS")
print("="*60)
print(f"Accuracy: {avg_acc:.4f}")
print(f"AUC-ROC:  {avg_auc:.4f}")
print("\nComparison to best single model:")
print(f"  Best single: {results['Accuracy'].max():.4f}")
print(f"  Ensemble:    {avg_acc:.4f}")
improvement = (avg_acc - results['Accuracy'].max()) * 100
print(f"  Improvement: {'+' if improvement > 0 else ''}{improvement:.2f}%")

### 3.3 Method 2: Weighted Averaging

In [ ]:
# Optimize weights using validation set approach
# For simplicity, we'll use test AUC as proxy for validation performance
# In practice, use a separate validation set!

# Calculate weights proportional to AUC scores
auc_scores = np.array([xgb_auc, rf_auc, lr_auc])
weights = auc_scores / auc_scores.sum()

print("="*60)
print("⚖️ OPTIMIZED WEIGHTS (based on AUC)")
print("="*60)
for model_name, weight, auc in zip(['XGBoost', 'Random Forest', 'Logistic Reg'], weights, auc_scores):
    print(f"{model_name:20s}: weight={weight:.3f}  (AUC={auc:.4f})")

# Weighted average
weighted_prob = weights[0] * xgb_prob + weights[1] * rf_prob + weights[2] * lr_prob
weighted_pred = (weighted_prob >= 0.5).astype(int)

weighted_acc = accuracy_score(y_test, weighted_pred)
weighted_auc = roc_auc_score(y_test, weighted_prob)

print("\n" + "="*60)
print("📊 WEIGHTED AVERAGING RESULTS")
print("="*60)
print(f"Accuracy: {weighted_acc:.4f}")
print(f"AUC-ROC:  {weighted_auc:.4f}")
print("\nComparison:")
print(f"  Best single model: {results['Accuracy'].max():.4f}")
print(f"  Simple averaging:  {avg_acc:.4f}")
print(f"  Weighted average:  {weighted_acc:.4f}")
improvement = (weighted_acc - results['Accuracy'].max()) * 100
print(f"  Improvement over best: {'+' if improvement > 0 else ''}{improvement:.2f}%")

### 3.4 Method 3: Stacking with Meta-Model

<div style='background: #f3e5f5; padding: 15px; border-left: 5px solid #9c27b0; border-radius: 5px; margin: 15px 0;'>
    <h4 style='margin: 0 0 10px 0; color: #4a148c;'>🏗️ Stacking Concept</h4>
    <p style='margin: 0; color: #4a148c;'>
        <strong>Simple averaging:</strong> All models weighted equally (or fixed weights)<br>
        <strong>Stacking:</strong> Train a meta-model to learn WHEN to trust each base model!<br><br>
        <strong>How it works:</strong><br>
        1. Base models predict on validation set<br>
        2. Meta-features = [XGB prediction, RF prediction, LR prediction]<br>
        3. Meta-model learns: "XGB good for X, RF good for Y"<br>
        4. Meta-model makes final decision<br><br>
        <strong>Key insight:</strong> Different models excel on different sample types!
    </p>
</div>

In [ ]:
# For proper stacking, we need to split train into train/validation
# to avoid overfitting the meta-model

# Split training data into train and validation for stacking
X_train_stack, X_val_stack, y_train_stack, y_val_stack = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, stratify=y_train
)

print("="*60)
print("🏗️ STACKING SETUP")
print("="*60)
print(f"Stack training set:   {len(X_train_stack):,} samples")
print(f"Stack validation set: {len(X_val_stack):,} samples")
print(f"Final test set:       {len(X_test):,} samples")

# Retrain base models on stacking training set
print("\nRetraining base models on stacking train set...")

xgb_stack = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, 
                          random_state=42, eval_metric='logloss')
xgb_stack.fit(X_train_stack, y_train_stack)

rf_stack = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_stack.fit(X_train_stack, y_train_stack)

lr_stack = LogisticRegression(max_iter=1000, random_state=42)
lr_stack.fit(X_train_stack, y_train_stack)

# Generate meta-features from validation set
print("Generating meta-features from validation set...")
meta_features_val = np.column_stack([
    xgb_stack.predict_proba(X_val_stack)[:, 1],
    rf_stack.predict_proba(X_val_stack)[:, 1],
    lr_stack.predict_proba(X_val_stack)[:, 1]
])

# Generate meta-features from test set
meta_features_test = np.column_stack([
    xgb_stack.predict_proba(X_test)[:, 1],
    rf_stack.predict_proba(X_test)[:, 1],
    lr_stack.predict_proba(X_test)[:, 1]
])

print("\nMeta-features shape:")
print(f"  Validation: {meta_features_val.shape}")
print(f"  Test:       {meta_features_test.shape}")
print("\n  Each row = [XGB probability, RF probability, LR probability]")

In [ ]:
# Train meta-model (Logistic Regression)
print("Training meta-model (Logistic Regression)...")
meta_model = LogisticRegression(random_state=42)
meta_model.fit(meta_features_val, y_val_stack)

# Meta-model learned weights
print("\n" + "="*60)
print("🧠 META-MODEL LEARNED WEIGHTS")
print("="*60)
print("\nCoefficients (how much to trust each base model):")
for model_name, coef in zip(['XGBoost', 'Random Forest', 'Logistic Reg'], 
                            meta_model.coef_[0]):
    print(f"  {model_name:20s}: {coef:+.4f}")
print(f"\n  Intercept: {meta_model.intercept_[0]:.4f}")

print("\n💡 Interpretation:")
print("  • Positive coefficient: Trust this model's high predictions")
print("  • Negative coefficient: Be skeptical of this model's high predictions")
print("  • Magnitude: How much weight to give")

# Make predictions
stack_prob = meta_model.predict_proba(meta_features_test)[:, 1]
stack_pred = meta_model.predict(meta_features_test)

stack_acc = accuracy_score(y_test, stack_pred)
stack_auc = roc_auc_score(y_test, stack_prob)

print("\n" + "="*60)
print("📊 STACKING RESULTS")
print("="*60)
print(f"Accuracy: {stack_acc:.4f}")
print(f"AUC-ROC:  {stack_auc:.4f}")

### 3.5 Final Ensemble Comparison

In [ ]:
# Create comprehensive comparison
final_results = pd.DataFrame({
    'Method': [
        'XGBoost (best single)',
        'Random Forest',
        'Logistic Regression',
        '---',
        'Simple Averaging',
        'Weighted Averaging',
        'Stacking (Meta-model)'
    ],
    'Accuracy': [
        xgb_acc,
        rf_acc,
        lr_acc,
        np.nan,
        avg_acc,
        weighted_acc,
        stack_acc
    ],
    'AUC-ROC': [
        xgb_auc,
        rf_auc,
        lr_auc,
        np.nan,
        avg_auc,
        weighted_auc,
        stack_auc
    ]
})

print("="*60)
print("🏆 FINAL ENSEMBLE COMPARISON")
print("="*60)
print(final_results.to_string(index=False))

# Highlight best
best_method = final_results.loc[final_results['Accuracy'].idxmax(), 'Method']
best_acc = final_results['Accuracy'].max()
improvement = (best_acc - xgb_acc) * 100

print("\n" + "="*60)
print("🎯 KEY FINDINGS")
print("="*60)
print(f"Best method: {best_method}")
print(f"Best accuracy: {best_acc:.4f}")
print(f"Improvement over best single model: {'+' if improvement > 0 else ''}{improvement:.2f}%")

print("\n💡 Typical improvements:")
print("  • Simple averaging:  +0.5% to +1.5%")
print("  • Weighted averaging: +1.0% to +2.0%")
print("  • Stacking:          +1.5% to +3.0%")
print("\n  Small percentages = HUGE impact in production!")
print("  Example: 1% improvement on 1M loans = 10,000 better decisions")

### 3.6 Visualization: Model Agreement Patterns

In [ ]:
# Analyze prediction patterns
prediction_df = pd.DataFrame({
    'XGBoost': xgb_prob,
    'Random Forest': rf_prob,
    'Logistic Reg': lr_prob,
    'Simple Avg': avg_prob,
    'Stacking': stack_prob,
    'Actual': y_test.values
})

# Scatter plot: XGBoost vs Random Forest predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: XGBoost vs Random Forest
scatter = axes[0].scatter(
    prediction_df['XGBoost'], 
    prediction_df['Random Forest'],
    c=prediction_df['Actual'],
    cmap='RdYlGn_r',
    alpha=0.6,
    edgecolors='black',
    linewidth=0.5
)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect agreement')
axes[0].set_xlabel('XGBoost Probability', fontsize=12)
axes[0].set_ylabel('Random Forest Probability', fontsize=12)
axes[0].set_title('Model Agreement: XGBoost vs Random Forest', fontsize=13)
axes[0].legend()
axes[0].grid(alpha=0.3)
plt.colorbar(scatter, ax=axes[0], label='Actual (0=No Default, 1=Default)')

# Plot 2: Simple Average vs Stacking
scatter2 = axes[1].scatter(
    prediction_df['Simple Avg'],
    prediction_df['Stacking'],
    c=prediction_df['Actual'],
    cmap='RdYlGn_r',
    alpha=0.6,
    edgecolors='black',
    linewidth=0.5
)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect agreement')
axes[1].set_xlabel('Simple Average Probability', fontsize=12)
axes[1].set_ylabel('Stacking Probability', fontsize=12)
axes[1].set_title('Ensemble Comparison: Simple Avg vs Stacking', fontsize=13)
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.colorbar(scatter2, ax=axes[1], label='Actual (0=No Default, 1=Default)')

plt.tight_layout()
plt.show()

print("="*60)
print("📊 INTERPRETATION")
print("="*60)
print("Plot 1 (XGBoost vs RF):")
print("  • Points on diagonal: Models agree")
print("  • Points off diagonal: Models disagree (ensemble opportunity!)")
print("  • Color: Red = actual default, Green = no default")
print("\nPlot 2 (Simple Avg vs Stacking):")
print("  • Stacking adjusts predictions based on learned patterns")
print("  • Points off diagonal show where meta-model differs")
print("  • Ideally: Red points move up, green points move down")

<div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 10px; color: white; margin: 30px 0;'>
    <h2 style='margin: 0; font-size: 28px;'>MODULE 4: Conclusion & Best Practices</h2>
    <p style='margin: 10px 0 0 0; opacity: 0.9;'>Summary, key takeaways, and practical guidelines</p>
    <p style='margin: 5px 0 0 0; font-size: 14px;'>⏱️ Duration: 10 minutes</p>
</div>

### 4.1 Final Leaderboard

In [ ]:
# Create final summary with all metrics
print("="*70)
print("🏆 FINAL MODEL LEADERBOARD")
print("="*70)

leaderboard = final_results.dropna().copy()
leaderboard = leaderboard.sort_values('Accuracy', ascending=False)
leaderboard['Rank'] = range(1, len(leaderboard)+1)
leaderboard = leaderboard[['Rank', 'Method', 'Accuracy', 'AUC-ROC']]

print(leaderboard.to_string(index=False))

print("\n" + "="*70)
print("💰 BUSINESS IMPACT CALCULATION")
print("="*70)

# Hypothetical business metrics
total_loans = 100000
avg_loan_amount = 10000
default_loss_rate = 0.7  # Lose 70% of loan amount on default

best_single_acc = xgb_acc
ensemble_acc = leaderboard.iloc[0]['Accuracy']
improvement_pct = (ensemble_acc - best_single_acc)

# Loans saved from wrong decisions
loans_improved = total_loans * improvement_pct
value_saved = loans_improved * avg_loan_amount * default_loss_rate

print(f"\nScenario: {total_loans:,} loan applications/year")
print(f"Average loan amount: ${avg_loan_amount:,}")
print(f"Loss on default: {default_loss_rate:.0%} of loan amount")
print(f"\nAccuracy improvement: {improvement_pct:.2%}")
print(f"Better decisions: ~{loans_improved:,.0f} loans/year")
print(f"\n💵 Estimated value saved: ${value_saved:,.0f}/year")
print("\n   Small accuracy gains = MASSIVE business impact!")

### 4.2 Key Takeaways

<div style='background: #e8f5e9; padding: 20px; border-left: 5px solid #4caf50; border-radius: 5px; margin: 20px 0;'>
    <h4 style='margin: 0 0 15px 0; color: #1b5e20;'>✅ What We Learned</h4>
    
<h4 style='color: #2e7d32; margin-top: 15px;'>SHAP Values (Interpretability)</h4>
<ul style='color: #1b5e20;'>
    <li><strong>Foundation:</strong> Based on game theory (Shapley values) - provably fair attribution</li>
    <li><strong>Property:</strong> Additive: Base value + Σ(SHAP values) = Final prediction</li>
    <li><strong>Global:</strong> Summary plot shows feature importance across all samples</li>
    <li><strong>Local:</strong> Waterfall plot explains individual predictions step-by-step</li>
    <li><strong>Use case:</strong> Regulatory compliance, customer communication, model debugging</li>
</ul>

<h4 style='color: #2e7d32; margin-top: 15px;'>Ensembling (Accuracy Boost)</h4>
<ul style='color: #1b5e20;'>
    <li><strong>Simple averaging:</strong> Easy to implement, typically +0.5-1.5% improvement</li>
    <li><strong>Weighted averaging:</strong> Optimized weights based on validation performance</li>
    <li><strong>Stacking:</strong> Meta-model learns when to trust each base model</li>
    <li><strong>Key requirement:</strong> Base models must be DIVERSE (different errors)</li>
    <li><strong>Impact:</strong> Small percentage gains = huge business value at scale</li>
</ul>
</div>

### 4.3 Decision Guide: When to Use What

| Scenario | Recommended Approach | Rationale |
|----------|---------------------|----------|
| **Need interpretability** | SHAP + Single model | Simpler to explain |
| **Need max accuracy** | Stacking ensemble | Best performance |
| **Limited compute** | Simple averaging | Fast, good enough |
| **Small dataset (<1K)** | Single model | Avoid overfitting |
| **Large dataset (>10K)** | Stacking | Enough data for meta-model |
| **Production latency critical** | Single model | Avoid multiple model calls |
| **Regulatory compliance** | SHAP explanations | Transparent decisions |
| **Model debugging** | SHAP waterfall plots | Understand specific errors |

### 4.4 Best Practices Checklist

<div style='background: #fff3e0; padding: 20px; border-left: 5px solid #ff9800; border-radius: 5px; margin: 20px 0;'>
    <h4 style='margin: 0 0 15px 0; color: #e65100;'>📋 Production Deployment Checklist</h4>
    
<strong style='color: #e65100;'>Interpretability:</strong>
<ul style='color: #e65100;'>
    <li>✓ Always validate SHAP explanations make business sense</li>
    <li>✓ Check for unexpected feature importance (potential data leakage)</li>
    <li>✓ Document how explanations are communicated to stakeholders</li>
    <li>✓ Monitor SHAP values over time for distribution shifts</li>
</ul>

<strong style='color: #e65100;'>Ensembling:</strong>
<ul style='color: #e65100;'>
    <li>✓ Verify base models are diverse (check error correlation)</li>
    <li>✓ Use separate validation set for meta-model training</li>
    <li>✓ Start simple (averaging) before complex (stacking)</li>
    <li>✓ Monitor individual model performance in production</li>
    <li>✓ Document ensemble architecture for maintainability</li>
</ul>

<strong style='color: #e65100;'>General:</strong>
<ul style='color: #e65100;'>
    <li>✓ Always use holdout test set for final evaluation</li>
    <li>✓ Track both accuracy AND business metrics</li>
    <li>✓ Version control models and explanation artifacts</li>
    <li>✓ A/B test before full deployment</li>
</ul>
</div>

### 4.5 Real-World Applications

**Finance & Banking:**
- Credit scoring with GDPR-compliant explanations
- Fraud detection with interpretable alerts to investigators
- Loan approval with customer-facing rejection reasons

**Healthcare:**
- Disease diagnosis with doctor-interpretable feature contributions
- Treatment recommendations with evidence-based reasoning
- Patient risk stratification with actionable insights

**E-commerce:**
- Product recommendations with "Why this product?" explanations
- Dynamic pricing with transparent factor analysis
- Customer churn prediction with retention action priorities

**Insurance:**
- Premium calculation with understandable risk factors
- Claim approval with documented reasoning
- Fraud investigation with highlighted suspicious patterns

### 4.6 Next Steps & Further Learning

**Deepen Your Knowledge:**
1. **SHAP Advanced Topics:**
   - Force plots (alternative to waterfall)
   - Dependence plots (feature interactions)
   - SHAP for neural networks (DeepSHAP)

2. **Advanced Ensembling:**
   - Blending vs Stacking trade-offs
   - Multi-level stacking
   - Boosting as sequential ensembling

3. **Production MLOps:**
   - Model monitoring and drift detection
   - A/B testing frameworks
   - Explanation caching strategies

**Recommended Resources:**
- SHAP Documentation: https://shap.readthedocs.io/
- Christoph Molnar's "Interpretable ML" book
- Kaggle ensemble tutorials
- Papers: "A Unified Approach to Interpreting Model Predictions" (SHAP paper)

<div style='background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%); padding: 30px; border-radius: 10px; color: white; margin: 40px 0; text-align: center;'>
    <h2 style='margin: 0; font-size: 32px;'>🎉 Congratulations!</h2>
    <p style='margin: 20px 0 10px 0; font-size: 18px; opacity: 0.95;'>
        You've mastered explainability and ensembling!
    </p>
    <p style='margin: 10px 0; opacity: 0.9;'>
        ✅ Understand SHAP values from game theory<br>
        ✅ Interpret global and local model explanations<br>
        ✅ Build ensemble models for improved accuracy<br>
        ✅ Apply these techniques to real-world problems
    </p>
    <p style='margin: 20px 0 0 0; font-size: 14px; opacity: 0.8;'>
        <em>"The best model is one you can both trust and explain."</em>
    </p>
</div>

---

<div style='text-align: center; padding: 20px; color: #666;'>
    <p style='margin: 0;'><strong>Vishlesan i-Hub IIT Patna × Masai School</strong></p>
    <p style='margin: 5px 0;'>AI & Machine Learning Program</p>
    <p style='margin: 5px 0; font-size: 12px;'>From Black Boxes to Glass Boxes, From Single Models to Super-Ensembles</p>
</div>